In [224]:
words = open('names.txt', 'r').read().splitlines()

In [225]:
# bigrams = 28 * 28 array with each element being an int repr freq of how many times 
# char at index i followed char at index i+1

In [226]:
import torch

In [227]:
N = torch.zeros((27, 27), dtype=torch.int32)

In [228]:
charToInd = dict()
indToChar = dict()

allChars = sorted(set(''.join(words)))
allChars = ['.'] + allChars

index = 0
for char in allChars:
    print(char, index)
    charToInd[char] = index
    indToChar[index] = char
    index += 1



. 0
a 1
b 2
c 3
d 4
e 5
f 6
g 7
h 8
i 9
j 10
k 11
l 12
m 13
n 14
o 15
p 16
q 17
r 18
s 19
t 20
u 21
v 22
w 23
x 24
y 25
z 26


In [229]:
for word in words:
    N[0, charToInd[word[0]]] += 1
    N[charToInd[word[-1]], 0] += 1
    for char, nextChar in zip(word, word[1:]):
        idx1, idx2 = charToInd[char], charToInd[nextChar]
        N[idx1, idx2] += 1

In [230]:
print(N)

tensor([[   0, 4410, 1306, 1542, 1690, 1531,  417,  669,  874,  591, 2422, 2963,
         1572, 2538, 1146,  394,  515,   92, 1639, 2055, 1308,   78,  376,  307,
          134,  535,  929],
        [6640,  556,  541,  470, 1042,  692,  134,  168, 2332, 1650,  175,  568,
         2528, 1634, 5438,   63,   82,   60, 3264, 1118,  687,  381,  834,  161,
          182, 2050,  435],
        [ 114,  321,   38,    1,   65,  655,    0,    0,   41,  217,    1,    0,
          103,    0,    4,  105,    0,    0,  842,    8,    2,   45,    0,    0,
            0,   83,    0],
        [  97,  815,    0,   42,    1,  551,    0,    2,  664,  271,    3,  316,
          116,    0,    0,  380,    1,   11,   76,    5,   35,   35,    0,    0,
            3,  104,    4],
        [ 516, 1303,    1,    3,  149, 1283,    5,   25,  118,  674,    9,    3,
           60,   30,   31,  378,    0,    1,  424,   29,    4,   92,   17,   23,
            0,  317,    1],
        [3983,  679,  121,  153,  384, 1271,   82,

In [231]:
sumN = torch.sum(N, 1, keepdim=True)
sumN
prob = N / sumN
# print(torch.sum(M, 1))

In [232]:
# test1 = torch.randint(1,3, (2,2))
# print(test1)
# sumTest = torch.sum(test1, dim=1)
# print(sumTest)
# print(test1 / sumTest)

In [233]:
index = 0
word = ''
while True:
    distribution = prob[index]
    next_index = torch.multinomial(distribution, num_samples=1, replacement=True).item()

    if next_index == 0:
        break
    word += indToChar[next_index]
    index = next_index
print(word)

an


In [234]:
# loss function calculation
word = 'vedanta'
# what is our models probability to predict anna
ix1 = 0
prob_word = 1
for char in word:
    ix2 = charToInd[char]
    prob_word *= prob[ix1, ix2]
    ix1 = ix2
prob_word *= prob[ix1, 0]
print(prob_word)

# that is a really small number, let us do log and addition instead
log_liklihood = 0
for char in word:
    ix2 = charToInd[char]
    log_liklihood += torch.log(prob[ix1, ix2])
    ix1 = ix2
log_liklihood += torch.log(prob[ix1, 0])
print(-log_liklihood/len(word))



tensor(1.6189e-09)
tensor(2.7859)


Now moving onto building the same bigram but from NN now

In [287]:
from torch.nn import functional as F
from torch import tensor

# forward pass
W = torch.randn((27,27), requires_grad=True)
# F.one_hot(torch.tensor([400,2,3]), num_classes=401)
# W

In [288]:
inputs, targets = [], []
for word in words:
    complete_word = '.'+word+'.'
    for char_input, char_target in zip(complete_word, complete_word[1:]):
        inputs.append(charToInd[char_input])
        targets.append(charToInd[char_target])

    # break

In [274]:
inputs

[0,
 5,
 13,
 13,
 1,
 0,
 15,
 12,
 9,
 22,
 9,
 1,
 0,
 1,
 22,
 1,
 0,
 9,
 19,
 1,
 2,
 5,
 12,
 12,
 1,
 0,
 19,
 15,
 16,
 8,
 9,
 1,
 0,
 3,
 8,
 1,
 18,
 12,
 15,
 20,
 20,
 5,
 0,
 13,
 9,
 1,
 0,
 1,
 13,
 5,
 12,
 9,
 1,
 0,
 8,
 1,
 18,
 16,
 5,
 18,
 0,
 5,
 22,
 5,
 12,
 25,
 14,
 0,
 1,
 2,
 9,
 7,
 1,
 9,
 12,
 0,
 5,
 13,
 9,
 12,
 25,
 0,
 5,
 12,
 9,
 26,
 1,
 2,
 5,
 20,
 8,
 0,
 13,
 9,
 12,
 1,
 0,
 5,
 12,
 12,
 1,
 0,
 1,
 22,
 5,
 18,
 25,
 0,
 19,
 15,
 6,
 9,
 1,
 0,
 3,
 1,
 13,
 9,
 12,
 1,
 0,
 1,
 18,
 9,
 1,
 0,
 19,
 3,
 1,
 18,
 12,
 5,
 20,
 20,
 0,
 22,
 9,
 3,
 20,
 15,
 18,
 9,
 1,
 0,
 13,
 1,
 4,
 9,
 19,
 15,
 14,
 0,
 12,
 21,
 14,
 1,
 0,
 7,
 18,
 1,
 3,
 5,
 0,
 3,
 8,
 12,
 15,
 5,
 0,
 16,
 5,
 14,
 5,
 12,
 15,
 16,
 5,
 0,
 12,
 1,
 25,
 12,
 1,
 0,
 18,
 9,
 12,
 5,
 25,
 0,
 26,
 15,
 5,
 25,
 0,
 14,
 15,
 18,
 1,
 0,
 12,
 9,
 12,
 25,
 0,
 5,
 12,
 5,
 1,
 14,
 15,
 18,
 0,
 8,
 1,
 14,
 14,
 1,
 8,
 0,
 12,
 9,
 12,
 12,
 9,
 1,
 1

In [290]:
num = len(inputs)

# gradient descent
for k in range(100):
  
  # forward pass
  xenc = F.one_hot(tensor(inputs), num_classes=27).float() # input to the network: one-hot encoding
  logits = xenc @ W # predict log-counts
  counts = logits.exp() # counts, equivalent to N
  probs = counts / counts.sum(1, keepdims=True) # probabilities for next character


  # loss by just normally looping
  # loss= 0
  # for i in range(5):
  #     prob = probs[i, targets[i]]
  #     loss -= torch.log(prob)
  # print(loss/5, loss2)
  loss = -probs[torch.arange(num), targets].log().mean() + 0.01*(W**2).mean()
  print(loss.item())
  
  # backward pass
  W.grad = None # set to zero the gradient
  loss.backward()
  
  # update
  W.data += -50 * W.grad

3.3696234226226807
3.16525936126709
3.034900665283203
2.9421770572662354
2.8722944259643555
2.8184871673583984
2.776266574859619
2.742440700531006
2.7147367000579834
2.691602945327759
2.6720094680786133
2.6552581787109375
2.640843629837036
2.628368377685547
2.6175100803375244
2.6080009937286377
2.5996201038360596
2.59218692779541
2.5855538845062256
2.579598903656006
2.574223041534424
2.5693435668945312
2.564892053604126
2.560811758041382
2.557056188583374
2.5535855293273926
2.5503671169281006
2.547372817993164
2.5445799827575684
2.5419681072235107
2.539520740509033
2.5372231006622314
2.535062789916992
2.5330278873443604
2.53110933303833
2.529297351837158
2.5275847911834717
2.5259642601013184
2.5244297981262207
2.5229740142822266
2.5215933322906494
2.5202815532684326
2.5190348625183105
2.517848253250122
2.516718864440918
2.5156424045562744
2.514615535736084
2.5136351585388184
2.5126986503601074
2.511803150177002
2.5109457969665527
2.5101253986358643
2.50933837890625
2.5085842609405518
2

In [ ]:
# encoded_input = F.one_hot(tensor(inputs), 27).float()
# W = torch.randn((27,27), requires_grad=True)
# logits = encoded_input @ W

# # softmax
# counts = logits.exp()
# probs = counts / counts.sum(1, keepdim=True)



# loss = -probs[torch.arange(len(inputs)), targets].log().mean() + 0.01*(W**2).mean()
# print(loss.item())


3.6056861877441406


In [ ]:
# W.grad = None
# loss.backward()
# W = W -50 * W.grad

TypeError: unsupported operand type(s) for *: 'float' and 'NoneType'

In [ ]:
counts = logits.exp()
probs = counts / torch.sum(counts, dim=1, keepdim=True)


tensor(3.3814) tensor(3.3814)


tensor(16.9069)

In [ ]:
# inputs, targets

([0, 5, 13, 13, 1], [5, 13, 13, 1, 0])

In [ ]:
# torch.sum(probs, dim=1, keepdim=True)

tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.]])

In [ ]:
# loss computation
probs

tensor(-19.5011)